# 💊 40 · Molecular machine learning for drug discovery
### Molecular representations (SMILES, descriptors, fingerprints, graphs), QSAR and graph models, why the train/test split decides what you measure, applicability domain and uncertainty, activity cliffs, recovering structure–activity rules, and virtual screening

**BiologyToML** · Level 2 · Research frontier · Part H — Genomes, proteins and molecules · Module 40 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/40_molecular_ml_drug_discovery.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, scikit-learn, PyTorch (CPU), Matplotlib |
| **Prerequisites** | Level 1 (especially Modules 07 ridge, 09 logistic regression, 10 random forests, 23 cross-validation and grouped splits); Module 28 (graph neural networks) for the message-passing baseline |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [07 · Ridge and lasso](07_ridge_lasso_sparse_regression.ipynb), [09 · Logistic regression](09_logistic_regression.ipynb), [10 · Decision trees and random forests](10_decision_trees_random_forests.ipynb), [23 · Model selection](23_model_selection.ipynb), [28 · Graph neural networks](28_graph_neural_networks.ipynb) |
| **Research frontier covered** | molecular representations (SMILES, fingerprints, graphs), QSAR and GNN property prediction, scaffold splits and generalisation, applicability domain and uncertainty, activity cliffs, virtual screening — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED** with the toolkit's `simulate_molecules`: a library of drug-like molecules built from 10
> ring scaffolds and 15 substituents, with **known, documented structure–activity rules** (a scaffold-specific baseline
> potency; an H-bond donor at site 0 adds +1.2 pIC50; a small lipophilic group at site 1 adds +0.5; a bulky group at
> site 2 subtracts 1.0; very lipophilic molecules lose potency) and a noise-free **oracle** for the true potency and
> solubility of any molecule. The SMILES strings are valid and RDKit-parsable. Real assays are noisier, molecules are
> larger and more diverse, and the "rules" are unknown — Section 10 explains what changes. Nothing here is a real drug
> or a clinical recommendation.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** the hit-to-lead drug-discovery pipeline and its assays (IC50/pIC50, solubility, ADMET) and **read** a SMILES string and the rule-of-five descriptors of a molecule.
2. **Represent** a molecule three ways — descriptor vector, circular fingerprint, graph — and **compute** Tanimoto similarity and nearest neighbours by hand and in code.
3. **Build** QSAR models (ridge, random forest, and a compact message-passing network from [Module 28](28_graph_neural_networks.ipynb)) and **report** R² and RMSE honestly.
4. **Demonstrate** that a **random** split overstates performance and a **scaffold** or **time** split measures generalisation to new chemistry, using grouped cross-validation ([Module 23](23_model_selection.ipynb)).
5. **Quantify** a model's applicability domain (error vs similarity to the training set) and **test** whether its predicted uncertainty tracks its error — motivating conformal prediction ([Module 42](42_trustworthy_ml.ipynb)).
6. **Find** activity cliffs (near-identical molecules with very different potency) and **show** why they break similarity-based models.
7. **Recover** the simulator's hidden structure–activity rules from a trained model and **check** them against the ground-truth oracle.
8. **Run** a virtual screen: rank a fresh library, compute the enrichment factor and ROC/PR for actives, and **filter** on a second objective (solubility).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · The pipeline and its assays | hits→leads, pIC50, SMILES, descriptors, rule of five | 20 min |
| 2 · Representations | descriptors, fingerprints, Tanimoto, nearest neighbours, graphs | 20 min |
| 3 · QSAR models | ridge, random forest, a compact MPNN; R² and RMSE | 20 min |
| 4 · 🧪 Splits decide what you measure | random vs scaffold vs time split; the gap | 25 min |
| 5 · Applicability domain and uncertainty | error vs similarity; do ensembles know when they are wrong? | 20 min |
| 6 · Activity cliffs | similar molecules, different potency; where models fail | 15 min |
| 7 · 🧪 Recovering the hidden rules | feature importances vs the documented SAR and the oracle | 15 min |
| 8 · Virtual screening | ranking, enrichment factor, ROC/PR, multi-objective filtering | 15 min |
| 9 · Multi-task learning | predicting potency and solubility together | 10 min |
| 10 · 🧭 From toy to real data | RDKit, ChEMBL, docking, antibiotics, what goes wrong | 8 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 7 min |

## 🧫 The biological question

Bringing a drug to market takes well over a decade and, by widely cited estimates, more than a billion dollars, and the
great majority of candidates fail — for lack of potency, poor solubility or absorption, toxicity, or simply because the
target turned out not to matter. Chemists cannot synthesise and test more than a tiny fraction of the possible drug-like
molecules (estimates of that space run to $10^{60}$). So the central question of computational drug discovery is a
prediction problem: **given molecules whose potency and properties have been measured, can we predict those of molecules
not yet made, well enough to decide what to synthesise next — and, just as important, can we tell when *not* to trust the
prediction?**

This is **QSAR** — quantitative structure–activity relationship — an idea that goes back to Hansch and Fujita in 1964 and
now spans everything from linear regression on a handful of descriptors to graph neural networks and molecular language
models. Two results frame the modern field:

* **It can work at scale.** Stokes *et al.* (2020) trained a message-passing neural network on about 2,300 molecules
  tested against *E. coli*, screened it over a large chemical library, and experimentally confirmed new antibacterial
  compounds, including **halicin** — a structure unlike known antibiotics. Wong *et al.* (2024) extended the approach and
  added interpretability.
* **It is easy to fool yourself.** The headline accuracy of a molecular model depends enormously on *how you split the
  data*. A random train/test split, which leaves near-identical analogues on both sides, routinely reports numbers that
  collapse under a **scaffold split** that asks the real question — will this work on a new chemical series? (Sheridan
  2013; van Tilborg *et al.* 2022).

This notebook builds both halves — useful prediction and honest evaluation — on a **simulated** library whose rules we
know exactly, so every claim can be checked against the ground truth. It is the drug-discovery counterpart of
[Module 28](28_graph_neural_networks.ipynb) (which introduced molecules as graphs) and leans on the model-selection
machinery of [Module 23](23_model_selection.ipynb).

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import average_precision_score, mean_squared_error, r2_score, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, KFold

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)              # PyTorch has its own random-number generator
torch.set_num_threads(1)            # the networks here are tiny; one thread avoids overhead (and is polite on shared CPUs)
T_START = time.time()               # to report the total run time at the end
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads(), "| device: cpu")

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the molecule simulator (library + a noise-free `oracle`), the SMILES writer, the descriptor and circular-fingerprint functions, `tanimoto` similarity and `assemble_molecule` (scaffold + substituents → graph). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

_AR = 1.5


MOLECULE_SCAFFOLDS = {
    "benzene": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)], [0, 1, 2, 3]),
    "pyridine": ([("N", 1)] + [("C", 1)] * 5, [(i, (i + 1) % 6, _AR) for i in range(6)], [2, 3, 4]),
    "pyrimidine": ([("N", 1), ("C", 1), ("N", 1), ("C", 1), ("C", 1), ("C", 1)],
                   [(i, (i + 1) % 6, _AR) for i in range(6)], [1, 3, 4]),
    "thiophene": ([("S", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "furan": ([("O", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "cyclohexane": ([("C", 0)] * 6, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 1, 2, 3]),
    "piperidine": ([("N", 0)] + [("C", 0)] * 5, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 2, 3]),
    "morpholine": ([("O", 0), ("C", 0), ("C", 0), ("N", 0), ("C", 0), ("C", 0)],
                   [(i, (i + 1) % 6, 1) for i in range(6)], [3, 1, 2]),
    "naphthalene": ([("C", 1)] * 10, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR), (5, 0, _AR),
                                      (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR), (9, 5, _AR)], [0, 1, 2, 7]),
    "quinoline": ([("N", 1)] + [("C", 1)] * 9, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR),
                                                (5, 0, _AR), (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR),
                                                (9, 5, _AR)], [1, 2, 7]),
}


MOLECULE_SUBSTITUENTS = {      # attachment atom = index 0
    "methyl": ([("C", 0)], []),
    "ethyl": ([("C", 0), ("C", 0)], [(0, 1, 1)]),
    "hydroxyl": ([("O", 0)], []),
    "amino": ([("N", 0)], []),
    "fluoro": ([("F", 0)], []),
    "chloro": ([("Cl", 0)], []),
    "methoxy": ([("O", 0), ("C", 0)], [(0, 1, 1)]),
    "carboxyl": ([("C", 0), ("O", 0), ("O", 0)], [(0, 1, 2), (0, 2, 1)]),
    "amide": ([("C", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 1)]),
    "nitrile": ([("C", 0), ("N", 0)], [(0, 1, 3)]),
    "trifluoromethyl": ([("C", 0), ("F", 0), ("F", 0), ("F", 0)], [(0, 1, 1), (0, 2, 1), (0, 3, 1)]),
    "sulfonamide": ([("S", 0), ("O", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 2), (0, 3, 1)]),
    "dimethylamino": ([("N", 0), ("C", 0), ("C", 0)], [(0, 1, 1), (0, 2, 1)]),
    "acetyl": ([("C", 0), ("O", 0), ("C", 0)], [(0, 1, 2), (0, 2, 1)]),
    "phenyl": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)]),
}


_VALENCE = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1}


_MASS = {"C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "H": 1.008}


def molecule_to_smiles(atoms, bonds) -> str:
    """Write a molecular graph as a SMILES string (depth-first, ring closures as digits; not canonical)."""
    n = len(atoms)
    nbr = {i: [] for i in range(n)}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    for i in nbr:
        nbr[i].sort()
    parent, order, closures, seen = {0: None}, [], {}, set()

    def visit(a):                                    # first pass: tree edges vs ring-closure edges
        seen.add(a)
        order.append(a)
        for b, o in nbr[a]:
            if b not in seen:
                parent[b] = a
                visit(b)
            elif b != parent[a] and (min(a, b), max(a, b)) not in closures:
                closures[(min(a, b), max(a, b))] = o
    visit(0)
    ring_at = {i: [] for i in range(n)}
    for k, (pair, o) in enumerate(closures.items()):
        for a in pair:
            ring_at[a].append((k + 1, o, pair))

    def bond_symbol(o, a, b):
        if o == 2:
            return "="
        if o == 3:
            return "#"
        return "-" if (o == 1 and atoms[a][1] and atoms[b][1]) else ""

    def write(a):
        el, arom = atoms[a]
        s = el.lower() if arom else el
        for digit, o, pair in ring_at[a]:
            first = a == min(pair, key=order.index)
            s += (bond_symbol(o, *pair) if first else "") + (str(digit) if digit < 10 else f"%{digit}")
        kids = [(b, o) for b, o in nbr[a] if parent.get(b) == a]
        for idx, (b, o) in enumerate(kids):
            sub = bond_symbol(o, a, b) + write(b)
            s += sub if idx == len(kids) - 1 else f"({sub})"
        return s
    return write(0)


def molecule_descriptors(atoms, bonds) -> dict:
    """Simple descriptors: heavy atoms, implicit H, molecular weight, H-bond donors/acceptors,
    aromatic atoms, rings and a simplified (Crippen-like) octanol–water partition coefficient."""
    used = np.zeros(len(atoms))
    for i, j, o in bonds:
        used[i] += o
        used[j] += o
    n_h = np.array([max(0, int(np.floor(_VALENCE[el] - used[k] + 1e-9))) if el in ("C", "N", "O") else 0
                    for k, (el, _) in enumerate(atoms)])
    els = [el for el, _ in atoms]
    mw = sum(_MASS[e] for e in els) + _MASS["H"] * n_h.sum()
    hbd = sum(1 for k, e in enumerate(els) if e in ("N", "O") and n_h[k] > 0)
    hba = sum(1 for e in els if e in ("N", "O"))
    contrib = {"C": 0.25, "N": -0.75, "O": -0.55, "S": 0.4, "F": 0.35, "Cl": 0.7}
    logp = sum(contrib[e] + (0.12 if a else 0) for e, a in atoms) - 0.3 * hbd
    return dict(heavy_atoms=len(atoms), n_hydrogens=int(n_h.sum()), mol_weight=round(mw, 2), hbd=hbd, hba=hba,
                aromatic_atoms=int(sum(a for _, a in atoms)), rings=len(bonds) - len(atoms) + 1,
                logp=round(logp, 3))


def molecular_fingerprint(atoms, bonds, radius: int = 2, n_bits: int = 1024) -> np.ndarray:
    """Circular (Morgan/ECFP-like) fingerprint: hashed atom environments up to ``radius`` bonds, as 0/1 bits."""
    import zlib
    nbr = {i: [] for i in range(len(atoms))}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    ids = [zlib.crc32(repr((el, ar, len(nbr[k]))).encode()) for k, (el, ar) in enumerate(atoms)]
    bits = np.zeros(n_bits, dtype=np.int8)
    for _ in range(radius + 1):
        for v in ids:
            bits[v % n_bits] = 1
        ids = [zlib.crc32(repr((ids[k], sorted((o, ids[j]) for j, o in nbr[k]))).encode()) for k in range(len(atoms))]
    return bits


def tanimoto(a, b) -> float:
    """Tanimoto (Jaccard) similarity of two binary fingerprints: |a ∧ b| / |a ∨ b|."""
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    union = np.sum(a | b)
    return float(np.sum(a & b) / union) if union else 0.0


def assemble_molecule(scaffold: str, substituents: dict):
    """Molecular graph (atoms, bonds) of ``scaffold`` carrying ``substituents`` = {site index: substituent name}."""
    atoms, bonds, sites = MOLECULE_SCAFFOLDS[scaffold]
    atoms, bonds = list(atoms), list(bonds)
    for s, name in sorted(substituents.items()):
        f_atoms, f_bonds = MOLECULE_SUBSTITUENTS[name]
        off = len(atoms)
        atoms += f_atoms
        bonds += [(i + off, j + off, o) for i, j, o in f_bonds] + [(sites[s], off, 1)]
    return atoms, bonds


def simulate_molecules(rng: np.random.Generator, n_molecules: int = 2000, noise: float = 0.3) -> dict:
    """Simulate a small-molecule library with a bioactivity (pIC50) and a solubility (logS) assay.

    Each molecule = one of 10 ring scaffolds + 1–3 substituents at the scaffold's attachment
    sites.  Hidden structure–activity rules (the "biology" a model must learn): each scaffold
    has its own baseline potency; a hydrogen-bond donor at site 0 (the key pocket) adds +1.2;
    small lipophilic groups at site 1 add +0.5; bulky groups at site 2 clash (−1.0); very
    lipophilic molecules lose potency (−0.4 per logP unit above 3); plus small random
    site × substituent effects.  Solubility falls with logP and size.

    Returns a dict with ``table`` (SMILES, scaffold, substituents, descriptors, measured
    ``pIC50`` and ``logS``), ``graphs`` (list of (atoms, bonds)), ``scaffold_effect``, and
    ``oracle(scaffold, substituents)`` → noise-free (pIC50, logS) of *any* design, e.g.
    ``oracle("pyridine", {0: "amino", 2: "methyl"})`` — the ground truth for design (Module 39).
    """
    scaffolds, subs = list(MOLECULE_SCAFFOLDS), list(MOLECULE_SUBSTITUENTS)
    weights = rng.dirichlet(np.full(len(scaffolds), 2.0))                 # some scaffolds are rare
    scaffold_effect = dict(zip(scaffolds, rng.normal(5.5, 0.8, len(scaffolds))))
    site_effect = rng.normal(0, 0.2, (4, len(subs)))
    donors = {"hydroxyl", "amino", "amide", "sulfonamide", "carboxyl"}
    lipophilic = {"methyl", "ethyl", "chloro", "trifluoromethyl"}
    bulky = {"phenyl", "sulfonamide", "dimethylamino", "trifluoromethyl"}

    def oracle(scaffold: str, substituents: dict):
        """Noise-free (pIC50, logS) of a scaffold + {site: substituent} design under the hidden rules."""
        desc = molecule_descriptors(*assemble_molecule(scaffold, substituents))
        act = scaffold_effect[scaffold] + sum(site_effect[s, subs.index(nm)] for s, nm in substituents.items())
        act += 1.2 * (substituents.get(0) in donors) + 0.5 * (substituents.get(1) in lipophilic) \
            - 1.0 * (substituents.get(2) in bulky) - 0.4 * max(0.0, desc["logp"] - 3.0)
        logs = 0.5 - 0.8 * desc["logp"] - 0.006 * (desc["mol_weight"] - 150) + 0.2 * desc["hbd"]
        return float(act), float(logs)

    rows, graphs, seen = [], [], set()
    while len(rows) < n_molecules:
        sc = scaffolds[rng.choice(len(scaffolds), p=weights)]
        n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
        k = int(rng.integers(1, min(3, n_sites) + 1))
        chosen = {int(s): subs[rng.integers(len(subs))] for s in sorted(rng.choice(n_sites, k, replace=False))}
        key = (sc, tuple(sorted(chosen.items())))
        if key in seen:
            continue
        seen.add(key)
        atoms, bonds = assemble_molecule(sc, chosen)
        act, logs = oracle(sc, chosen)
        rows.append(dict(smiles=molecule_to_smiles(atoms, bonds), scaffold=sc,
                         substituents="; ".join(f"site{s}:{nm}" for s, nm in sorted(chosen.items())),
                         **molecule_descriptors(atoms, bonds),
                         pIC50=act + rng.normal(0, noise), logS=logs + rng.normal(0, noise)))
        graphs.append((atoms, bonds))
    return dict(table=pd.DataFrame(rows), graphs=graphs, scaffold_effect=scaffold_effect, oracle=oracle)


print("🧰 Toolkit loaded:", ", ".join(['molecule_to_smiles', 'molecule_descriptors', 'molecular_fingerprint', 'tanimoto', 'assemble_molecule', 'simulate_molecules']))

---
## 1 · The drug-discovery pipeline and its assays

> 📍 **Where we are.** Before any model, we need to know what is being predicted and how it is measured. This section
> introduces the pipeline, the potency assay, how molecules are written as text (SMILES), the descriptors a chemist reads
> at a glance, and Lipinski's rule of five.

🔬 **Assay box — from target to candidate.** A drug project starts with a **target** (often a protein whose activity is
linked to disease). High-throughput screening of large compound collections yields **hits** — molecules that show some
activity. Medicinal chemists then make analogues to turn hits into **leads** (more potent, more selective) and leads into
**candidates** (potent *and* soluble, absorbable, metabolically stable and non-toxic — the **ADMET** properties).

The central potency measure is the **IC50**: the concentration of compound that halves the target's activity. Because it
spans many orders of magnitude it is reported as $\mathrm{pIC50}=-\log_{10}(\mathrm{IC50}/\text{1 M})$, so pIC50 $=6$
means IC50 $=1\,\mu$M and pIC50 $=9$ means 1 nM; **one pIC50 unit is a ten-fold change in potency**, and higher is more
potent. We also track **solubility**, reported as $\log S$ (log of the molar solubility; higher is more soluble).

Our simulated library returns measured `pIC50` and `logS` (with assay noise) plus the molecule's structure and
descriptors. The hidden rules that generate the potency are documented in the simulator — we will try to rediscover them
in Section 7 and check against them throughout.

In [ ]:
# ── 1.1 A SIMULATED compound library ──
rng = np.random.default_rng(SEED)
lib = simulate_molecules(rng, 2000)              # 2,000 molecules (fast; the toolkit default is also small)
mols = lib["table"]                              # one row per molecule
graphs = lib["graphs"]                           # list of (atoms, bonds) — the molecular graphs
oracle = lib["oracle"]                           # oracle(scaffold, {site: substituent}) → noise-free (pIC50, logS)
print("SIMULATED library:", len(mols), "molecules,", mols.scaffold.nunique(), "scaffolds")
print(mols[["smiles", "scaffold", "substituents", "mol_weight", "logp", "hbd", "hba", "pIC50", "logS"]].head(6).to_string())
print(f"\npIC50 range {mols.pIC50.min():.2f}–{mols.pIC50.max():.2f}; "
      f"'actives' (pIC50 > 7, i.e. IC50 < 100 nM): {(mols.pIC50 > 7).mean():.1%} of the library")

**Reading a SMILES string.** SMILES (Weininger 1988) writes a molecule as text. Atoms are their element symbols
(`C`, `N`, `O`, `S`, `F`, `Cl`); **aromatic** atoms are lower-case (`c`, `n`, `o`, `s`); single, double and triple bonds
are nothing, `=` and `#`; **branches** are in parentheses; and **ring closures** use matching digits. So `c1ccccc1` is
benzene (six aromatic carbons closing a ring labelled `1`), `c1ccncc1` is pyridine, and `C(=O)N` is an amide (a carbon
double-bonded to O and single-bonded to N). Hydrogens on C, N, O are implicit — filled in to complete each atom's
valence.

In [ ]:
# ── 1.2 Dissect a few SMILES, and the descriptors behind them ──
for k in [0, 1, 4]:
    atoms, bonds = graphs[k]
    d = molecule_descriptors(atoms, bonds)
    print(f"{mols.smiles[k]:28s} {mols.scaffold[k]:11s}  heavy={d['heavy_atoms']:2d} "
          f"MW={d['mol_weight']:6.1f} logP={d['logp']:+.2f} HBD={d['hbd']} HBA={d['hba']} "
          f"aromatic={d['aromatic_atoms']} rings={d['rings']}")
print("\n(heavy = non-hydrogen atoms; MW = molecular weight; logP = lipophilicity; "
      "HBD/HBA = hydrogen-bond donors/acceptors)")

In [ ]:
# ── 1.3 Lipinski's rule of five: a drug-likeness filter (Lipinski et al. 1997) ──
d = pd.DataFrame([molecule_descriptors(a, b) for a, b in graphs])
violations = ((d.mol_weight > 500).astype(int) + (d.logp > 5) + (d.hbd > 5) + (d.hba > 10))
print("Lipinski's rule of five — an orally-available drug usually has:")
print("  MW ≤ 500, logP ≤ 5, H-bond donors ≤ 5, H-bond acceptors ≤ 10")
print(f"\nin this SIMULATED library:")
print(f"  pass all four rules : {(violations == 0).mean():.1%}")
print(f"  ≥ 2 violations      : {(violations >= 2).mean():.1%}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, col, cut, name in zip(axes, ["mol_weight", "logp", "hbd"], [500, 5, 5],
                              ["molecular weight", "logP", "H-bond donors"]):
    ax.hist(d[col], bins=25, color="tab:blue", alpha=0.8)
    ax.axvline(cut, color="tab:red", ls="--", lw=1.2, label=f"Ro5 limit = {cut}")
    ax.set_xlabel(name); ax.set_ylabel("molecules"); ax.legend(fontsize=7)
fig.suptitle("SIMULATED library vs Lipinski's rule of five", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Almost the whole simulated library satisfies the rule of five — by construction, the molecules
are small and drug-like. The rule of five is a **soft filter**, not a law: it summarises properties of orally-available
drugs known in 1997, and plenty of real drugs (and most injected biologics) break it. It is a useful first screen, the
kind a model's predictions must be combined with before deciding what to make.

⚠️ **pIC50 is a log scale.** A model with an RMSE of 1.0 in pIC50 is, on average, off by a *factor of ten* in potency —
often the difference between a lead and a dead end. Always read regression errors on molecules in these physical terms,
not as abstract numbers.

✏️ **Your turn.** Pick the most potent molecule in the library (`mols.loc[mols.pIC50.idxmax()]`) and the least potent,
print their SMILES and descriptors, and note one structural difference. (We will explain *which* differences matter in
Section 7.)

---
## 2 · Representations: how a molecule becomes a feature vector

> 📍 **Where we are.** Every model of Level 1 expects a fixed-length vector of numbers. A molecule is a variable-size
> graph. Three representations bridge the gap; each makes different things easy.

**1. Descriptor vectors.** A handful of interpretable numbers (molecular weight, logP, counts of donors, acceptors,
rings, aromatic atoms, …). Compact and human-readable, but they throw away *where* groups are.

**2. Circular fingerprints (ECFP / Morgan).** A long bit vector in which each bit marks the presence of a particular
**atom environment** — an atom together with its neighbours out to a fixed radius. Rogers & Hahn (2010) formalised this
as repeatedly hashing each atom's identity together with its neighbours' current hashes (the same message-passing idea as
[Module 28](28_graph_neural_networks.ipynb), but with fixed hashing instead of learned weights) and setting one bit per
environment. Fingerprints are the workhorse of QSAR: fast, and good at "does this substructure occur?".

**3. Graphs.** Atoms and bonds fed to a graph neural network, which *learns* the representation
([Module 28](28_graph_neural_networks.ipynb)). The most flexible, the most data-hungry.

🧮 **Math refresher — Tanimoto similarity.** For two fingerprints (bit sets) $a$ and $b$,

$$\mathrm{Tanimoto}(a,b)=\frac{|a\cap b|}{|a\cup b|}=\frac{\text{bits on in both}}{\text{bits on in either}} .$$

*In words:* the number of environments the two molecules share, divided by the number either one has — the Jaccard index
of their bit sets. It runs from 0 (nothing in common) to 1 (identical fingerprints). A Tanimoto above about 0.7 (on the
usual ECFP) is often taken as "similar"; it is the basis of similarity search, clustering, and the **similar-property
principle** — the (imperfect!) assumption that similar molecules have similar activity (Section 6 shows where it breaks).

In [ ]:
# ── 2.1 One molecule, three representations ──
k = 4
atoms, bonds = graphs[k]
print("molecule:", mols.smiles[k], f"({mols.scaffold[k]}, {mols.substituents[k]})")
print("\n1. descriptor vector:", {kk: v for kk, v in molecule_descriptors(atoms, bonds).items()})
fp_k = molecular_fingerprint(atoms, bonds)                 # 1024-bit circular fingerprint
print(f"\n2. circular fingerprint: {len(fp_k)} bits, {fp_k.sum()} set; indices on =", list(np.flatnonzero(fp_k))[:12], "…")
print(f"\n3. graph: {len(atoms)} atoms, {len(bonds)} bonds (fed to the MPNN of Module 28 in Section 3)")

In [ ]:
# ── 2.2 Fingerprints + Tanimoto: find a molecule's nearest neighbours ──
fp = np.array([molecular_fingerprint(a, b) for a, b in graphs])     # 2000 × 1024 bit matrix
print(f"fingerprint matrix {fp.shape}; a molecule has {fp.sum(1).mean():.0f} bits on, on average")

query = 4
sims = np.array([tanimoto(fp[query], fp[j]) for j in range(len(fp))])
sims[query] = -1                                            # ignore the molecule itself
neighbours = np.argsort(-sims)[:4]
print(f"\nquery: {mols.smiles[query]}  (pIC50 {mols.pIC50[query]:.2f})")
print("nearest neighbours by Tanimoto similarity:")
for j in neighbours:
    print(f"  sim {sims[j]:.3f}  {mols.smiles[j]:26s} pIC50 {mols.pIC50[j]:.2f}  ({mols.scaffold[j]})")
print(f"\nmedian nearest-neighbour similarity across the library: "
      f"{np.median([np.sort([tanimoto(fp[i], fp[j]) for j in rng.choice(len(fp), 200, replace=False)])[-2] for i in rng.choice(len(fp), 150, replace=False)]):.2f}")

🔍 **What just happened?** The query's nearest neighbours are molecules that share most of its atom environments — same
scaffold, similar substituents — and they tend to have similar potency, which is exactly why fingerprint models work at
all. But note two things we will return to: a molecule can have a neighbour at Tanimoto 1.0 that is **not identical** (the
fingerprint is lossy — it can miss *where* a group sits), and similar neighbours do **not always** have similar potency
(activity cliffs, Section 6).

⚠️ **Fingerprints are lossy and collide.** Hashing many possible environments into 1,024 bits means different
environments can set the same bit (a collision), and two different molecules can share a fingerprint. Larger fingerprints
(2,048+ bits) reduce collisions; RDKit's real ECFP also records counts and stereochemistry. Our toolkit fingerprint is a
teaching version — good enough to learn the ideas, not a production descriptor.

✏️ **Your turn.** Compute the Tanimoto similarity between two molecules on the *same* scaffold that differ in one
substituent, and between two on *different* scaffolds. By how much does one substituent change the fingerprint compared
with changing the whole ring system?

---
## 3 · QSAR models: ridge, random forest, and a graph network

> 📍 **Where we are.** With fingerprints we can reuse the regression models of Level 1 directly, and with graphs the
> message-passing network of [Module 28](28_graph_neural_networks.ipynb). We fit three and compare them on a **random**
> train/test split (Section 4 revisits the split itself).

* **Ridge regression** ([Module 07](07_ridge_lasso_sparse_regression.ipynb)) on the fingerprint bits — a linear model,
  regularised because there are 1,024 features and many are rare.
* **Random forest** ([Module 10](10_decision_trees_random_forests.ipynb)) on the fingerprint bits — captures
  interactions between substructures, the standard strong QSAR baseline.
* **A compact MPNN** — the sum-pooling message-passing network of [Module 28](28_graph_neural_networks.ipynb), which
  learns the representation from the graph instead of using fixed fingerprint bits.

The **noise ceiling** is the $R^2$ of the oracle's noise-free potency against the measured potency: because the assay adds
noise, no model can systematically do better than this.

In [ ]:
# ── 3.1 Featurise molecules for the MPNN (same scheme as Module 28) ──
ATOM_ELEMENTS = ["C", "N", "O", "S", "F", "Cl"]
VALENCE_OF = {"C": 4, "N": 3, "O": 2}
N_MAX = max(len(a) for a, _ in graphs)


def featurize_molecule(atoms, bonds, n_max=N_MAX):
    """Padded atom-feature matrix (n_max × 15), adjacency (n_max × n_max) and a 0/1 mask of real atoms."""
    n = len(atoms)
    bond_sum, degree = np.zeros(n), np.zeros(n, int)
    A = np.zeros((n_max, n_max), np.float32)
    for i, j, order in bonds:
        A[i, j] = A[j, i] = 1
        bond_sum[[i, j]] += order
        degree[[i, j]] += 1
    X = np.zeros((n_max, 15), np.float32)
    for a_i, (el, aromatic) in enumerate(atoms):
        X[a_i, ATOM_ELEMENTS.index(el)] = 1
        X[a_i, 6] = aromatic
        X[a_i, 6 + min(degree[a_i], 4)] = 1                  # columns 7–10: number of heavy neighbours
        n_h = max(0, int(np.floor(VALENCE_OF.get(el, 0) - bond_sum[a_i] + 1e-9)))
        X[a_i, 11 + min(n_h, 3)] = 1                         # columns 11–14: number of hydrogens
    mask = np.zeros(n_max, np.float32)
    mask[:n] = 1
    return X, A, mask


feats = [featurize_molecule(a, b) for a, b in graphs]
X_mol = torch.tensor(np.stack([f[0] for f in feats]))
A_mol = torch.tensor(np.stack([f[1] for f in feats]))
M_mol = torch.tensor(np.stack([f[2] for f in feats]))
y = mols.pIC50.values.astype(np.float32)
print("MPNN tensors:", tuple(X_mol.shape), tuple(A_mol.shape), tuple(M_mol.shape))

In [ ]:
# ── 3.2 The compact message-passing network (sum pooling), from Module 28 ──
class MolecularMPNN(nn.Module):
    """h_i ← ReLU(W_self h_i + W_nbr Σ_j A_ij h_j), repeated; sum readout; MLP → one number per molecule."""

    def __init__(self, d_in=15, d_hidden=64, n_layers=3):
        super().__init__()
        self.embed = nn.Linear(d_in, d_hidden)
        self.self_lin = nn.ModuleList(nn.Linear(d_hidden, d_hidden) for _ in range(n_layers))
        self.nbr_lin = nn.ModuleList(nn.Linear(d_hidden, d_hidden, bias=False) for _ in range(n_layers))
        self.head = nn.Sequential(nn.Linear(d_hidden, d_hidden), nn.ReLU(), nn.Linear(d_hidden, 1))

    def forward(self, X, A, mask):
        H = F.relu(self.embed(X)) * mask[..., None]
        for s_lin, n_lin in zip(self.self_lin, self.nbr_lin):
            H = F.relu(s_lin(H) + A @ n_lin(H)) * mask[..., None]
        return self.head(H.sum(dim=1)).squeeze(-1)           # SUM readout → permutation-invariant


def train_mpnn(train_idx, target=y, epochs=40, batch=64, seed=SEED):
    """Mini-batch Adam on standardised target; returns predictions for every molecule."""
    torch.manual_seed(seed)
    g = np.random.default_rng(seed)
    model = MolecularMPNN()
    opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-3)
    mu, sd = target[train_idx].mean(), target[train_idx].std()
    y_t = torch.tensor((target - mu) / sd)
    for _ in range(epochs):
        model.train()
        perm = g.permutation(train_idx)
        for b in range(0, len(perm), batch):
            ib = torch.tensor(perm[b:b + batch])
            opt.zero_grad()
            F.mse_loss(model(X_mol[ib], A_mol[ib], M_mol[ib]), y_t[ib]).backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        return model(X_mol, A_mol, M_mol).numpy() * sd + mu


print("MPNN parameters:", sum(p.numel() for p in MolecularMPNN().parameters()))

In [ ]:
# ── 3.3 🧪 Ground-truth check: three QSAR models on a RANDOM split ──
t0 = time.time()
parse_subs = lambda s: {int(p.split(":")[0][4:]): p.split(":")[1] for p in s.split("; ")}
y_oracle = np.array([oracle(sc, parse_subs(s))[0] for sc, s in zip(mols.scaffold, mols.substituents)])
perm = np.random.default_rng(SEED).permutation(len(mols))
tr, te = np.sort(perm[400:]), np.sort(perm[:400])
preds = {"fingerprint + ridge": Ridge(alpha=10.0).fit(fp[tr], y[tr]).predict(fp[te]),
         "fingerprint + random forest": RandomForestRegressor(200, max_features=0.3, random_state=SEED, n_jobs=1)
         .fit(fp[tr], y[tr]).predict(fp[te]),
         "graph MPNN": train_mpnn(tr)[te],
         "noise ceiling (oracle)": y_oracle[te]}
qsar = pd.DataFrame({name: {"R2": r2_score(y[te], p), "RMSE": mean_squared_error(y[te], p) ** 0.5}
                     for name, p in preds.items()}).T.round(3)
print(qsar); print(f"\n({time.time() - t0:.0f} s)")

In [ ]:
# ── 3.4 Predicted vs measured potency (random forest) ──
rf_pred = preds["fingerprint + random forest"]
fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.scatter(y[te], rf_pred, s=10, alpha=0.5, color="tab:blue")
lims = [y[te].min() - 0.3, y[te].max() + 0.3]
ax.plot(lims, lims, "k--", lw=0.8, label="perfect")
ax.set_xlabel("measured pIC50 (SIMULATED)"); ax.set_ylabel("predicted pIC50")
ax.set_title(f"random forest, random split\nR² = {r2_score(y[te], rf_pred):.2f}, "
             f"RMSE = {mean_squared_error(y[te], rf_pred) ** 0.5:.2f} pIC50 units")
ax.legend(fontsize=8); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On a random split all three models land in the same range (R² roughly 0.5–0.6), comfortably
better than guessing the mean but well short of the noise ceiling (R² ≈ 0.9): the hidden rules are site-specific
("a donor *at site 0*") and the models must learn them separately for each scaffold from a few dozen examples. The MPNN
edges out the fingerprint models here, but not dramatically — on data sets this size a well-tuned random forest on
fingerprints is a famously hard baseline to beat (Wu *et al.* 2018; Yang *et al.* 2019). The RMSE of roughly 0.7 pIC50
units means predictions are typically off by a factor of about five in IC50.

⚠️ **The model is only as honest as its split.** Every number above comes from a *random* split, in which the test
molecules closely resemble training molecules. That is almost never the question drug discovery asks. Section 4 shows what
happens when we ask a fairer one.

✏️ **Your turn.** Change `max_features` of the random forest from 0.3 to `"sqrt"` and 1.0 and re-run 3.3. Which value
generalises best, and how does it relate to the de-correlation argument of [Module 10](10_decision_trees_random_forests.ipynb)?

### 📄 Research connection — From QSAR to deep molecular machine learning

- **What the field does.** Quantitative structure–activity relationships began with Hansch & Fujita (1964), who regressed biological activity on
physicochemical descriptors such as logP. Circular fingerprints (Morgan 1965; formalised as ECFP by Rogers & Hahn
2010) made substructure-based models the field's workhorse. Deep learning arrived with multi-task networks (Mayr
*et al.* 2016) and message-passing graph networks benchmarked by MoleculeNet (Wu *et al.* 2018) and refined by
Chemprop's directed MPNN (Yang *et al.* 2019).
- **What this notebook builds.** Ridge and random-forest QSAR on the toolkit's circular fingerprint, and the compact sum-pooling MPNN of Module 28,
all scored against a known noise ceiling. Production descriptors (RDKit ECFP with counts and stereochemistry) and
Chemprop's directed message passing are richer, but the ideas are identical.
- **Where the frontier is.** As of 2025, pretrained molecular representations (self-supervised GNNs and SMILES/graph language models) and careful
data curation are the active frontiers, alongside a sobering body of work showing that on honest splits deep models
often only match or modestly beat tuned fingerprint baselines — so strong baselines and realistic evaluation matter
as much as architecture.
- **Read next.** Rogers & Hahn 2010 (*J Chem Inf Model*), *Extended-connectivity fingerprints*; Wu *et al.* 2018 (*Chem Sci*),
*MoleculeNet: a benchmark for molecular machine learning*; Yang *et al.* 2019 (*J Chem Inf Model*), *Analyzing
learned molecular representations for property prediction* (Chemprop).

---
## 4 · 🧪 Splits decide what you measure

> 📍 **Where we are.** [Module 23](23_model_selection.ipynb) warned that the test set must be *independent* of the
> training set and that grouped data break ordinary cross-validation. Molecules are the textbook case: libraries are full
> of close analogues, so a random split leaks near-duplicates across the train/test line and flatters the model.

Three ways to split, asking three different questions:

* **Random split** — "how well do we predict analogues of molecules we have already measured?" Easy; usually optimistic.
* **Scaffold split** — hold out *entire ring systems* (a Bemis–Murcko-style grouping; here we group by the simulator's
  `scaffold`). Asks "will the model work on a new chemical series?" This is **grouped $k$-fold cross-validation**
  ([Module 23](23_model_selection.ipynb)) with the scaffold as the group.
* **Time split** — train on "older", test on "newer" molecules. Asks "will a model trained on today's data predict what
  we make next?" Our library has no timestamps, so we simulate a project that makes **larger molecules over time** (lead
  optimisation tends to grow a molecule) and split on size.

A model that scores well on a random split but poorly on a scaffold split has learned the chemistry it has already seen,
not transferable rules.

In [ ]:
# ── 4.1 Scaffold split = grouped k-fold; time split = sort by size ──
t0 = time.time()


def cv_predictions(X, model_fn, groups=None, n_splits=4):
    """Out-of-fold predictions with KFold (random) or GroupKFold (scaffold). Returns R² and RMSE over all molecules."""
    splitter = GroupKFold(n_splits) if groups is not None else KFold(n_splits, shuffle=True, random_state=SEED)
    pred = np.zeros(len(X))
    for tr_i, te_i in splitter.split(X, groups=groups):
        pred[te_i] = model_fn().fit(X[tr_i], y[tr_i]).predict(X[te_i])
    return r2_score(y, pred), mean_squared_error(y, pred) ** 0.5


ridge_fn = lambda: Ridge(alpha=10.0)
rf_fn = lambda: RandomForestRegressor(120, max_features=0.3, random_state=SEED, n_jobs=1)
cv = {}                                               # cache so Section 4.2 can reuse these results
rows = []
for name, fn in [("ridge", ridge_fn), ("random forest", rf_fn)]:
    cv[(name, "random")] = cv_predictions(fp, fn, groups=None)
    cv[(name, "scaffold")] = cv_predictions(fp, fn, groups=mols.scaffold.values)
    rows.append(dict(model=name, split="random (4-fold)", R2=cv[(name, "random")][0], RMSE=cv[(name, "random")][1]))
    rows.append(dict(model=name, split="scaffold (grouped 4-fold)", R2=cv[(name, "scaffold")][0], RMSE=cv[(name, "scaffold")][1]))
print(pd.DataFrame(rows).round(3).to_string(index=False)); print(f"({time.time() - t0:.0f} s)")

In [ ]:
# ── 4.2 A simulated TIME split: train on smaller (earlier) molecules, test on larger (later) ones ──
discovery_order = np.argsort(mols.heavy_atoms.values + np.random.default_rng(SEED).normal(0, 0.3, len(mols)))
early, late = discovery_order[:1600], discovery_order[1600:]
print(f"early molecules: mean {mols.heavy_atoms.values[early].mean():.1f} heavy atoms; "
      f"late: {mols.heavy_atoms.values[late].mean():.1f}")
for name, fn in [("ridge", ridge_fn), ("random forest", rf_fn)]:
    p = fn().fit(fp[early], y[early]).predict(fp[late])
    print(f"  {name:14s} time split: R² = {r2_score(y[late], p):.3f}, RMSE = {mean_squared_error(y[late], p) ** 0.5:.3f}")

# collect the random forest R² for the three splits into one bar chart (reuse the cached CV results)
rf_rand = cv[("random forest", "random")][0]
rf_scaf = cv[("random forest", "scaffold")][0]
rf_time = r2_score(y[late], rf_fn().fit(fp[early], y[early]).predict(fp[late]))
fig, ax = plt.subplots(figsize=(5.2, 3.2))
bars = ax.bar(["random", "time / size", "scaffold"], [rf_rand, rf_time, rf_scaf],
              color=["tab:green", "tab:orange", "tab:red"])
ax.axhline(0, color="k", lw=0.6); ax.set_ylabel("R² (random forest, SIMULATED)")
ax.set_ylim(min(-0.45, rf_scaf - 0.15), max(rf_rand, rf_time) + 0.12)
ax.set_title("the same model, three splits:\nhow you split sets what you measure")
for b, v in zip(bars, [rf_rand, rf_time, rf_scaf]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.02 if v > 0 else v - 0.03, f"{v:.2f}",
            ha="center", va="bottom" if v > 0 else "top", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The identical random forest reports a comfortably positive R² on the **random** split, a lower
one on the **time/size** split (it extrapolates to larger molecules than it trained on), and a **negative** R² on the
**scaffold** split — worse than just predicting the test-set mean. The held-out scaffolds have their own baseline
potency, and nothing in the training data reveals it, so the model's confident fingerprint-based guesses are
systematically wrong. (Exactly how negative depends on *which* scaffolds land in the held-out folds; rerun with a
different seed to watch it move.)

This is the single most important practical lesson of molecular ML: **report the split that matches your deployment
question.** If you will screen a library of familiar chemistry, a random split may be fair; if you want genuinely new
chemotypes, only a scaffold or time split tells you the truth. Papers that report only random-split numbers routinely
overstate what their model will do prospectively (Sheridan 2013).

⚠️ **Near-duplicates are leakage even within a "random" split.** Two molecules differing by one methyl are nearly the same
measurement; if one is in train and one in test, the test is not really independent. Real pipelines deduplicate by
similarity before splitting — and watch for the same molecule appearing in multiple assays.

✏️ **Your turn.** Add the MPNN to the scaffold comparison: fit it on one grouped fold (`train_mpnn` on the training
indices of a `GroupKFold` split) and compare its held-out R² with the random forest's. Does the graph model generalise
across scaffolds any better? (It usually does not, by much — architecture rarely rescues a hard split.)

---
## 5 · Applicability domain and uncertainty

> 📍 **Where we are.** Section 4 showed that a model fails on unfamiliar chemistry. A deployed model must therefore say
> *when* a prediction is outside its competence — its **applicability domain** — and ideally attach an **uncertainty** to
> each prediction. Can it?

Two complementary ideas:

* **Distance to the training set.** A prediction for a molecule close (high Tanimoto) to many training molecules should be
  more reliable than one for a molecule far from everything seen. We test this by plotting error against the maximum
  Tanimoto similarity to the training set.
* **Ensemble disagreement.** A random forest is an ensemble of trees ([Module 10](10_decision_trees_random_forests.ipynb));
  the *spread* of their predictions is a cheap uncertainty estimate. We ask whether that spread actually tracks the error.

To make distance vary over a wide range, we train on 70 % of the molecules from most scaffolds and test on a pool that
mixes the **held-out molecules of those scaffolds** (near) with molecules of **two entirely held-out scaffolds** (far).

In [ ]:
# ── 5.1 Build an in-distribution + out-of-distribution test pool ──
held_scaffolds = ["furan", "quinoline"]
in_dist = ~mols.scaffold.isin(held_scaffolds).values
idx_in = np.flatnonzero(in_dist)
rng_ad = np.random.default_rng(SEED)
rng_ad.shuffle(idx_in)
cut = int(0.7 * len(idx_in))
train_ad = idx_in[:cut]
test_ad = np.concatenate([idx_in[cut:], np.flatnonzero(~in_dist)])      # near (same scaffolds) + far (new scaffolds)

rf_ad = RandomForestRegressor(300, max_features=0.3, random_state=SEED, n_jobs=1).fit(fp[train_ad], y[train_ad])
pred_ad = rf_ad.predict(fp[test_ad])
abs_err = np.abs(pred_ad - y[test_ad])
tree_preds = np.stack([t.predict(fp[test_ad]) for t in rf_ad.estimators_])
tree_sd = tree_preds.std(0)                                             # ensemble disagreement = uncertainty


def max_similarity_to_train(j, train_fp):
    """Largest Tanimoto similarity of molecule j to any training molecule."""
    inter = (fp[j] & train_fp).sum(1)
    union = (fp[j] | train_fp).sum(1)
    return float((inter / np.maximum(union, 1)).max())


train_fp = fp[train_ad]
max_sim = np.array([max_similarity_to_train(j, train_fp) for j in test_ad])
print(f"test pool: {len(test_ad)} molecules, similarity-to-train from {max_sim.min():.2f} to {max_sim.max():.2f}")
print(f"correlation(error, 1 − max similarity): {np.corrcoef(abs_err, 1 - max_sim)[0, 1]:+.2f}")
print(f"correlation(error, tree disagreement) : {np.corrcoef(abs_err, tree_sd)[0, 1]:+.2f}")

In [ ]:
# ── 5.2 Error vs applicability domain, and does predicted uncertainty flag errors? ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
edges = [0.3, 0.45, 0.6, 0.8, 1.001]
centres, rmses = [], []
for lo, hi in zip([0.0] + edges[:-1], edges):
    m = (max_sim >= lo) & (max_sim < hi)
    if m.sum() > 3:
        centres.append((lo + hi) / 2); rmses.append((abs_err[m] ** 2).mean() ** 0.5)
a1.plot(centres, rmses, "o-", color="tab:red")
a1.set_xlabel("max Tanimoto similarity to training set"); a1.set_ylabel("RMSE (pIC50)")
a1.set_title("applicability domain:\nerror grows as molecules leave known chemistry"); a1.invert_xaxis()

q_hi, q_lo = np.quantile(tree_sd, 0.75), np.quantile(tree_sd, 0.25)
rmse_hi = (abs_err[tree_sd >= q_hi] ** 2).mean() ** 0.5
rmse_lo = (abs_err[tree_sd <= q_lo] ** 2).mean() ** 0.5
a2.scatter(tree_sd, abs_err, s=8, alpha=0.4, color="tab:blue")
a2.set_xlabel("ensemble disagreement (sd of trees)"); a2.set_ylabel("|prediction − truth| (pIC50)")
a2.set_title(f"does uncertainty track error?\nRMSE: high-uncertainty {rmse_hi:.2f} vs low {rmse_lo:.2f}")
plt.tight_layout(); plt.show()
print(f"RMSE on the most-uncertain quartile: {rmse_hi:.2f}   least-uncertain quartile: {rmse_lo:.2f}")

🔍 **What just happened?** The RMSE climbs steadily as molecules get **less similar** to the training set — from well under
one pIC50 unit for familiar chemistry to well over one for the held-out scaffolds. A simple maximum-Tanimoto cutoff is
therefore a usable **applicability-domain** flag: "we have seen nothing like this; distrust the number." Both signals are
only **modest** per molecule — the correlations with the absolute error are about +0.17 (for 1 − similarity) and +0.20
(for ensemble disagreement), so neither predicts an individual error precisely. They are useful mainly *in aggregate*:
the distance flag gives the clean, monotone binned curve on the left, and the most-uncertain quartile of molecules has a
clearly higher RMSE than the least-uncertain quartile (right panel). Random-forest and deep-ensemble variances are known
to be **miscalibrated**, especially out of distribution (the trees can agree confidently on the wrong answer), which is
why the per-molecule signal is this weak.

This is exactly why **conformal prediction** exists: it converts any model's scores into prediction *intervals* with a
guaranteed coverage rate, without assuming the uncertainty is calibrated.
[Module 42](42_trustworthy_ml.ipynb) builds it; for now, the lesson is that "the model is confident" and "the model is
right" are different claims, and distance-to-training data is a cheap, honest first guard.

⚠️ **A predicted value without a domain check is dangerous.** The scaffold-split disaster of Section 4 is invisible if you
only look at predictions; it becomes obvious the moment you notice the test molecules are far from everything you trained
on. Always report the similarity of your query set to your training set.

✏️ **Your turn.** Replace tree disagreement with a 5-member *bootstrap* ensemble of ridge regressions (fit each on a
resample of the training set; use the spread of their predictions). Does a linear ensemble's uncertainty track error any
better than the forest's?

---
## 6 · Activity cliffs: where similarity-based models break

> 📍 **Where we are.** Fingerprint models rest on the **similar-property principle**: similar molecules have similar
> activity. **Activity cliffs** are its counterexamples — pairs of very similar molecules with very *different* potency —
> and they are where models (and chemists) get caught out.

Our simulator manufactures cliffs on purpose. Recall the rule "an H-bond donor **at site 0** adds +1.2". Two molecules
carrying the same donor group but at different sites (site 0 versus site 1) are almost identical — often **identical in
fingerprint** — yet differ by more than a pIC50 unit. A model that sees only the fingerprint *cannot* tell them apart, so
it must be wrong about at least one.

In [ ]:
# ── 6.1 Find activity cliffs: high Tanimoto similarity, large potency difference ──
t0 = time.time()
rng_c = np.random.default_rng(SEED)
pairs = []
for i in rng_c.choice(len(mols), 800, replace=False):               # sample molecules, take each one's nearest neighbour
    sims = (fp[i] & fp).sum(1) / np.maximum((fp[i] | fp).sum(1), 1)
    sims[i] = -1
    j = int(sims.argmax())
    pairs.append((i, j, float(sims[j]), abs(y[i] - y[j])))
pairs = pd.DataFrame(pairs, columns=["i", "j", "similarity", "delta_pIC50"])
cliffs = pairs[(pairs.similarity >= 0.9) & (pairs.delta_pIC50 >= 1.5)]
print(f"of {len(pairs)} nearest-neighbour pairs, {len(cliffs)} are activity cliffs "
      f"(similarity ≥ 0.9 and ΔpIC50 ≥ 1.5)")

ex = cliffs.sort_values("delta_pIC50").iloc[-1]
i, j = int(ex.i), int(ex.j)
print(f"\nan activity cliff at Tanimoto similarity {ex.similarity:.2f}:")
for k in (i, j):
    print(f"  {mols.smiles[k]:26s} pIC50 {mols.pIC50[k]:.2f}  ({mols.scaffold[k]}; {mols.substituents[k]})")
print(f"  → same fingerprint, ΔpIC50 = {ex.delta_pIC50:.2f}: the donor sits at a different site (the +1.2 'site 0' rule)")

In [ ]:
# ── 6.2 A model's error is much larger on cliff members ──
rf_cliff = RandomForestRegressor(200, max_features=0.3, random_state=SEED, n_jobs=1).fit(fp[tr], y[tr])
err_all = np.abs(rf_cliff.predict(fp) - y)
cliff_members = np.unique(np.concatenate([cliffs.i.values, cliffs.j.values]).astype(int))
print(f"random-forest mean |error| on activity-cliff molecules: {err_all[cliff_members].mean():.2f} pIC50 units")
print(f"                              on the whole library     : {err_all.mean():.2f} pIC50 units")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
a1.scatter(pairs.similarity, pairs.delta_pIC50, s=10, alpha=0.4, color="0.5")
a1.scatter(cliffs.similarity, cliffs.delta_pIC50, s=16, color="tab:red", label="activity cliff")
a1.axhline(1.5, color="tab:red", ls=":", lw=0.8); a1.axvline(0.9, color="tab:red", ls=":", lw=0.8)
a1.set_xlabel("Tanimoto similarity to nearest neighbour"); a1.set_ylabel("|ΔpIC50| to nearest neighbour")
a1.set_title("most near-neighbours are similar in potency;\na few are cliffs (top right)"); a1.legend(fontsize=7)
a2.hist([err_all[cliff_members], err_all], bins=20, label=["cliff members", "all molecules"],
        color=["tab:red", "tab:blue"], density=True, alpha=0.7)
a2.set_xlabel("random-forest |error| (pIC50)"); a2.set_ylabel("density")
a2.set_title("the model is systematically worse on cliffs"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** Most nearest-neighbour pairs lie along the bottom of the left panel — similar molecules, similar
potency, as the similar-property principle promises. The handful of **cliffs** in the top-right corner violate it, and the
random forest's error on cliff molecules is several-fold larger than on the library as a whole. The example is the sharpest
case: two molecules with **identical fingerprints** but a potency difference of more than a log unit, because a
hydrogen-bond donor sits in the key pocket (site 0) in one and off to the side in the other. No fingerprint model can
resolve this; it would need to know *where* the group is — which a 3-D or carefully featurised graph model might, but our
lossy fingerprint cannot.

⚠️ **Cliffs inflate and deflate the wrong metrics.** Averaged R² can look fine while the model is useless exactly where it
matters — distinguishing a potent analogue from its inactive twin. Specialised approaches (pairwise "Δ-learning" models,
cliff-aware metrics) exist; van Tilborg *et al.* (2022) show that modern deep models do not reliably beat classical ones
on cliffs.

✏️ **Your turn.** Among the cliffs, how many are "fingerprint-identical" (similarity exactly 1.0)? For one such pair,
confirm with `molecule_descriptors` that the two molecules really are different despite the identical fingerprint.

### 📄 Research connection — Generalisation, splits and activity cliffs

- **What the field does.** How we measure a molecular model decides what we learn about it. Bemis & Murcko (1996) defined the molecular
scaffolds behind scaffold splits; Sheridan (2013) showed that random, scaffold (leave-class-out) and time splits give
very different, and increasingly realistic, error estimates; Tropsha (2010) codified QSAR validation practice. van
Tilborg, Alenicheva & Grisoni (2022) showed that activity cliffs remain hard even for modern deep models.
- **What this notebook builds.** A direct comparison of random, scaffold (grouped $k$-fold) and time splits on one model (Section 4), an
applicability-domain analysis (Section 5), and an activity-cliff analysis that ties the model's failures back to the
simulator's site-specific rule (Section 6).
- **Where the frontier is.** As of 2025: how to design splits that reflect real prospective use (temporal, assay-transfer, target-transfer);
cliff-aware training and evaluation; detecting and preventing subtle leakage (near-duplicates, the same compound
across assays); and whether reported deep-learning gains survive honest, leakage-controlled benchmarks.
- **Read next.** Sheridan 2013 (*J Chem Inf Model*), *Time-split cross-validation as a method for estimating the goodness of
prospective prediction*; Bemis & Murcko 1996 (*J Med Chem*), *The properties of known drugs: 1. Molecular
frameworks*; van Tilborg, Alenicheva & Grisoni 2022 (*J Chem Inf Model*), *Exposing the limitations of molecular
machine learning with activity cliffs*.

---
## 7 · 🧪 Can we recover the hidden structure–activity rules?

> 📍 **Where we are.** A model that predicts well is useful; a model we can *read* is more useful still, because a chemist
> can act on "add a donor here, remove the bulky group there". Here we ask the question the simulator lets us answer
> exactly: **does a trained model rediscover the documented rules?**

Interpreting raw fingerprint-bit importances is awkward because the hashing collides bits (Section 2), so we take the
cleaner route a chemist would: fit a transparent model on **interpretable features** — one indicator per (site,
substituent) combination — and read its feature importances
([Module 10](10_decision_trees_random_forests.ipynb)). Then we check the effects directly against the oracle's documented
rules: a hydrogen-bond donor at site 0 adds **+1.2**, a bulky group at site 2 subtracts **1.0**, a small lipophilic group
at site 1 adds **+0.5**.

In [ ]:
# ── 7.1 Fit a transparent model on site × substituent indicators ──
# MOLECULE_SUBSTITUENTS was copied into this notebook with the toolkit (it backs simulate_molecules)
SUBS = list(MOLECULE_SUBSTITUENTS)                                     # the substituent names (interpretable features)
DONORS = {"hydroxyl", "amino", "amide", "sulfonamide", "carboxyl"}     # the simulator's documented groups
BULKY = {"phenyl", "sulfonamide", "dimethylamino", "trifluoromethyl"}
LIPO = {"methyl", "ethyl", "chloro", "trifluoromethyl"}

design = np.zeros((len(mols), 4 * len(SUBS)))                          # indicator: substituent present at a given site
feat_names = [f"site{st}:{nm}" for st in range(4) for nm in SUBS]
for k, s in enumerate(mols.substituents):
    for site, nm in parse_subs(s).items():
        design[k, site * len(SUBS) + SUBS.index(nm)] = 1

rf_interp = RandomForestRegressor(300, max_features=0.3, random_state=SEED, n_jobs=1).fit(design, y)
importance = pd.Series(rf_interp.feature_importances_, index=feat_names).sort_values(ascending=False)
print("most important (site, substituent) features for potency:")
print(importance.head(8).round(4).to_string())
print("\ntop 5 features are all site-0 donors?",
      all(f.startswith("site0") and f.split(":")[1] in DONORS for f in importance.head(5).index))

In [ ]:
# ── 7.2 🧪 Ground-truth check: measured effects vs the documented rules and the oracle ──
has_donor0 = np.array([parse_subs(s).get(0) in DONORS for s in mols.substituents])
has_bulky2 = np.array([parse_subs(s).get(2) in BULKY for s in mols.substituents])
has_lipo1 = np.array([parse_subs(s).get(1) in LIPO for s in mols.substituents])
print("effect on measured pIC50 (difference of group means):")
print(f"  donor at site 0  : {y[has_donor0].mean() - y[~has_donor0].mean():+.2f}   (documented rule: +1.2)")
print(f"  bulky at site 2  : {y[has_bulky2].mean() - y[~has_bulky2].mean():+.2f}   (documented rule: −1.0)")
print(f"  lipophilic site 1: {y[has_lipo1].mean() - y[~has_lipo1].mean():+.2f}   (documented rule: +0.5)")

# cleanest check: ask the ORACLE to toggle one group on a fixed molecule (everything else held constant)
base_scaffold = "benzene"
for site, group, rule in [(0, "amino", "+1.2 (donor at site 0)"), (2, "phenyl", "−1.0 (bulky at site 2)")]:
    without = oracle(base_scaffold, {3: "methyl"})[0]
    with_group = oracle(base_scaffold, {3: "methyl", site: group})[0]
    print(f"  oracle: adding {group} at site {site} to benzene-3-methyl → ΔpIC50 = {with_group - without:+.2f}   [{rule}]")

🔍 **What just happened?** The transparent model's most important features are exactly the **site-0 donors** the simulator
rewards, and the group-mean differences recover the documented rules in sign and roughly in size: a donor at site 0 raises
potency by about a pIC50 unit, a bulky group at site 2 lowers it, a lipophilic group at site 1 helps a little. The group
means are not identical to the rule constants because substituents correlate (a bulky group is often also lipophilic) and
the baseline differs by scaffold — a reminder that **correlation-based importances mix direct and indirect effects**, just
like the regression coefficients of [Module 06](06_linear_regression_gwas.ipynb). The cleanest estimate comes from the
**oracle**: toggle one group on an otherwise-fixed molecule and read off the causal effect — a controlled "experiment"
that matches the documented rules in sign and roughly in size (amino at site 0 prints ≈ +1.3 vs the +1.2 rule). The
phenyl-at-site-2 toggle prints ≈ −1.3 rather than the −1.0 bulk rule because phenyl is also lipophilic: it pushes logP
above 3 and so triggers the simulator's extra −0.4-per-logP-unit penalty, adding about −0.3 on top of the bulk term — a
neat illustration that even a "controlled" single change can move *two* rules at once.

This is a (simulated) best case: the features *are* the mechanism. On real data the rules are unknown, descriptors are
entangled, and interpretability methods (bit importances, attention, SHAP, matched molecular pairs) give *hypotheses* to
test at the bench, not proven mechanisms — the same caution as a learned network edge
([Module 17](17_bayesian_networks_inference_learning.ipynb)).

✏️ **Your turn.** Use the oracle to measure the site-0 donor effect on a *different* scaffold — compare
`oracle("pyridine", {0: "amino", 1: "methyl"})` with `oracle("pyridine", {1: "methyl"})`. Is the donor effect the same
+1.2 as on benzene? What does that tell you about whether the rule is additive across scaffolds? (Note each scaffold
exposes only a few attachment sites, numbered 0, 1, 2, …)

---
## 8 · Virtual screening: ranking a library to decide what to make

> 📍 **Where we are.** In practice a model is rarely used to predict an exact pIC50; it is used to **rank** a large library
> so that experiments focus on the most promising molecules. The right metrics are therefore about the *top of the list*,
> not the average error.

We define **actives** as molecules with pIC50 > 7 (IC50 < 100 nM), train a model on the training set, rank the held-out
molecules by predicted potency, and measure:

* **Enrichment factor at 1 % (EF1%)** — among the top 1 % ranked molecules, how many times more actives than in the library
  as a whole. EF1% $=10$ means "ten-fold enriched": screening the top 1 % finds ten times as many actives as picking at
  random.
* **ROC-AUC and PR-AUC** — how well the ranking separates actives from inactives overall; PR-AUC is the more honest number
  when actives are rare ([Module 23](23_model_selection.ipynb)).

Then we add a second objective — solubility — because the most potent molecule is useless if it will not dissolve.

In [ ]:
# ── 8.1 Rank the held-out molecules and measure enrichment ──
active = (y > 7).astype(int)                                           # "actives": pIC50 > 7 (IC50 < 100 nM)
scores = rf_cliff.predict(fp)                                         # reuse the random-forest model from Section 6
rank = np.argsort(-scores[te])                                       # held-out molecules, best predicted first
base_rate = active[te].mean()
top1 = max(1, int(0.01 * len(te)))
ef1 = active[te][rank[:top1]].mean() / base_rate
ef5 = active[te][rank[:max(1, int(0.05 * len(te)))]].mean() / base_rate
print(f"held-out set: {len(te)} molecules, {active[te].sum()} actives ({base_rate:.1%})")
print(f"enrichment factor  EF1% = {ef1:.1f}×   EF5% = {ef5:.1f}×")
print(f"ROC-AUC = {roc_auc_score(active[te], scores[te]):.3f}   PR-AUC = {average_precision_score(active[te], scores[te]):.3f}")

In [ ]:
# ── 8.2 ROC curve and a potency–solubility (multi-objective) view ──
logS = mols.logS.values
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
fpr, tpr, _ = roc_curve(active[te], scores[te])
a1.plot(fpr, tpr, color="tab:blue", label=f"AUC = {roc_auc_score(active[te], scores[te]):.2f}")
a1.plot([0, 1], [0, 1], "k--", lw=0.8); a1.set_xlabel("false-positive rate"); a1.set_ylabel("true-positive rate")
a1.set_title("ranking actives (pIC50 > 7) in the held-out set"); a1.legend(fontsize=8)

top = te[rank[:top1 * 5]]                                             # the top 5 % the model would pick
a2.scatter(y[te], logS[te], s=8, alpha=0.3, color="0.6", label="held-out library")
a2.scatter(y[top], logS[top], s=22, color="tab:red", label="model's top 5 %")
a2.axhline(-4, color="tab:blue", ls=":", lw=1); a2.axvline(7, color="tab:green", ls=":", lw=1)
a2.text(7.05, a2.get_ylim()[0] + 0.3, "pIC50 = 7", color="tab:green", fontsize=7, rotation=90, va="bottom")
a2.text(a2.get_xlim()[0] + 0.2, -3.9, "logS = −4 (poorly soluble below)", color="tab:blue", fontsize=7)
a2.set_xlabel("measured pIC50"); a2.set_ylabel("measured logS (solubility)")
a2.set_title("multi-objective: potent AND soluble"); a2.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()

potent_soluble = top[(y[top] > 7) & (logS[top] > -4)]
print(f"of the model's top 5 % ({len(top)} molecules), {((y[top] > 7)).sum()} are truly active and "
      f"{len(potent_soluble)} are both active and reasonably soluble (logS > −4)")

🔍 **What just happened?** The model's ranking is enriched at the top: the top 1 % of the held-out set contains a
few-fold more actives than the base rate (though with only ~400 held-out molecules the top 1 % is a handful of
compounds, so EF1% here rests on a tiny denominator and is noisy — EF5% and the AUCs are steadier), and the ROC-AUC is
high. This is the metric that matters for screening — a
model with a mediocre R² can still be very useful if it reliably puts actives near the top. The potency–solubility plot
adds the second objective: here most of the model's top picks happen to be **both** potent and reasonably soluble (the red
points sit in the upper-right), which is good news — but that is not guaranteed. Potency and solubility trade off in this
simulator (lipophilic groups raise potency *and* lower solubility, a real medicinal-chemistry tension visible as the
downward tilt of the full library), so a candidate must clear *both* bars and real pipelines rank on a multi-objective
score or a Pareto front across potency, solubility and ADMET/toxicity predictions
([Module 39](39_generative_design.ipynb) optimises such trade-offs directly).

⚠️ **Enrichment depends on the base rate and the library.** EF1% is larger when actives are rarer, and a screen is only as
good as the library it ranks — a model cannot find chemotypes absent from the collection. Enrichment on a library full of
analogues of known actives (easy) is not enrichment on a diverse library (hard).

✏️ **Your turn.** Recompute EF1% using the **ridge** model's scores instead of the random forest's. Does the model with the
better R² also give the better top-1 % enrichment? (They need not agree — ranking and regression are different goals.)

### 📄 Research connection — AI-driven discovery in practice

- **What the field does.** Machine-learned rankings now feed real campaigns. Structure-based **ultra-large virtual screening** docks hundreds of
millions of molecules (Lyu *et al.* 2019); ligand-based deep models discovered the antibiotic halicin (Stokes *et al.*
2020) and a structural class active against *Acinetobacter baumannii* (Wong *et al.* 2024). The Therapeutics Data
Commons (Huang *et al.* 2021) collects standardised ADMET and activity benchmarks; enrichment and early-recognition
metrics such as BEDROC (Truchon & Bayly 2007) judge the top of the ranked list.
- **What this notebook builds.** A virtual screen on the simulated library: ranking, enrichment factor, ROC/PR, and a two-objective potency–solubility
filter. Real campaigns combine such rankings with docking, medicinal-chemistry triage and synthesis constraints.
- **Where the frontier is.** As of 2025: combining ligand-based ML with docking and physics; scaling to billions of make-on-demand molecules while
controlling false positives; prospective, blinded evaluation (a model is only proven by compounds it picked *before*
they were tested); and designing for synthesisability, not just predicted potency.
- **Read next.** Stokes *et al.* 2020 (*Cell*), *A deep learning approach to antibiotic discovery*; Wong *et al.* 2024 (*Nature*),
*Discovery of a structural class of antibiotics with explainable deep learning*; Lyu *et al.* 2019 (*Nature*),
*Ultra-large library docking for discovering new chemotypes*.

---
## 9 · Multi-task learning: potency and solubility together

> 📍 **Where we are.** Drug projects measure many endpoints on overlapping sets of molecules. **Multi-task learning**
> trains one network with a shared trunk and several output "heads", hoping that learning related tasks together builds a
> better shared representation — the neural analogue of borrowing strength across tasks.

We train a small network on fingerprints with a shared trunk and compare: (a) a **single-task** model for pIC50, and
(b) a **multi-task** model that predicts pIC50 *and* logS from the same trunk. Does the auxiliary solubility task help
potency?

In [ ]:
# ── 9.1 Single-task vs multi-task network on fingerprints ──
t0 = time.time()
Xfp = torch.tensor(fp.astype(np.float32))
y_pic, y_logs = y, mols.logS.values.astype(np.float32)


def train_shared(multitask, train_idx, test_idx, epochs=120, seed=SEED):
    """Shared trunk + 1 or 2 heads; returns R² for pIC50 on the test set."""
    torch.manual_seed(seed)
    trunk = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
    head_p, head_s = nn.Linear(64, 1), nn.Linear(64, 1)
    params = list(trunk.parameters()) + list(head_p.parameters()) + (list(head_s.parameters()) if multitask else [])
    opt = torch.optim.Adam(params, lr=1e-3, weight_decay=1e-3)
    mp, sp = y_pic[train_idx].mean(), y_pic[train_idx].std()
    ms, ss = y_logs[train_idx].mean(), y_logs[train_idx].std()
    Yp, Ys = torch.tensor((y_pic - mp) / sp), torch.tensor((y_logs - ms) / ss)
    g = np.random.default_rng(seed)
    for _ in range(epochs):
        for b in range(0, len(train_idx), 64):
            ib = torch.tensor(g.permutation(train_idx)[b:b + 64])
            opt.zero_grad()
            h = trunk(Xfp[ib])
            loss = F.mse_loss(head_p(h).squeeze(-1), Yp[ib])
            if multitask:
                loss = loss + F.mse_loss(head_s(h).squeeze(-1), Ys[ib])
            loss.backward(); opt.step()
    with torch.no_grad():
        pred = head_p(trunk(Xfp[test_idx])).squeeze(-1).numpy() * sp + mp
    return r2_score(y_pic[test_idx], pred)


single = np.mean([train_shared(False, tr, te, seed=s) for s in range(2)])
multi = np.mean([train_shared(True, tr, te, seed=s) for s in range(2)])
print(f"pIC50 R² on the held-out set (mean of 2 seeds):")
print(f"  single-task (pIC50 only)       : {single:.3f}")
print(f"  multi-task  (pIC50 + logS)     : {multi:.3f}")
print(f"({time.time() - t0:.0f} s)")

🔍 **What just happened?** On this library the auxiliary solubility task gives **essentially no improvement** to potency
prediction — the two numbers are within noise of each other. That is an honest and common outcome: multi-task learning
helps when the tasks genuinely share structure (and especially when the extra task has labels where the main task does
not), and it can even *hurt* when they do not ("negative transfer"). Here potency is driven by the site-specific SAR rules
while solubility is mostly a function of size and logP, so there is little to share.

In real ADMET modelling the picture is often friendlier: endpoints are correlated, each is measured on a different, partly
overlapping subset of molecules, and a shared representation lets a task borrow strength from the others (Mayr *et al.*
2016; the premise of Chemprop's multi-task mode). Whether to go multi-task is an empirical, per-dataset question — exactly
the kind of model-selection decision of [Module 23](23_model_selection.ipynb).

✏️ **Your turn.** Make the tasks share more by predicting pIC50 together with a *potency-related* auxiliary — the number of
H-bond donors (`mols.hbd`), which the site-0 rule ties to activity. Does multi-task help now?

---
## 10 · 🧭 From toy to real data

> 📍 **Where we are.** Everything above ran in about two minutes on a simulated library with known rules. Real molecular ML
> is bigger, noisier and full of traps. This section is reading, not code: what changes, what to use, and what goes wrong.

**Tools.** [RDKit](https://www.rdkit.org) is the foundation — it parses SMILES, computes real ECFP fingerprints and
descriptors, finds Bemis–Murcko scaffolds and enumerates matched molecular pairs. [DeepChem](https://deepchem.io) and
[Chemprop](https://chemprop.readthedocs.io) provide featurisers and trained message-passing models;
[scikit-learn](https://scikit-learn.org) (which we used) remains the workhorse for fingerprint models. We deliberately used
*none* of RDKit, DeepChem, Chemprop or PyTorch Geometric and wrote the pieces by hand so the mechanism is visible; the
toolkit's SMILES are valid and RDKit-parsable, so you can drop the library into RDKit as a next step.

**Public data.** Bioactivity: [ChEMBL](https://www.ebi.ac.uk/chembl/) (millions of measured activities),
[PubChem BioAssay](https://pubchem.ncbi.nlm.nih.gov), [BindingDB](https://www.bindingdb.org). Benchmarks:
[MoleculeNet](https://moleculenet.org), the [Therapeutics Data Commons](https://tdcommons.ai) and
[Polaris](https://polarishub.io) (which curates leakage-controlled splits). Make-on-demand chemical space:
[ZINC](https://zinc.docking.org) and Enamine REAL (billions of molecules).

**The first things that go wrong.**

1. **Optimistic splits and leakage.** Random splits leak near-duplicate analogues across train/test; the same compound
   appears in multiple assays; "scaffold" splits done loosely still share substructure. Use scaffold/time splits,
   deduplicate by similarity, and treat a suspiciously high R² as a bug to investigate (Section 4).
2. **Assay noise and censored values.** Measured IC50s carry experimental error (often several-fold) and are frequently
   **censored** ("> 10 µM" when a molecule is inactive at the top dose). Treating a censored value as a number, or merging
   measurements from different labs and protocols, injects noise and bias that no architecture fixes.
3. **Overtrusting a confident prediction.** A model will happily extrapolate far outside its applicability domain with no
   warning (Sections 4–5). Always report similarity-to-training-set, use calibrated uncertainty or conformal intervals
   ([Module 42](42_trustworthy_ml.ipynb)), and remember that the proof of a model is **prospective** — compounds it picked
   before they were tested.

Other realities: 3-D effects and stereochemistry (two mirror-image molecules with different activity) that a 2-D
fingerprint ignores; synthesisability (a predicted hit you cannot make is worthless); and the biosecurity and dual-use
questions that come with models that *design* molecules ([Module 39](39_generative_design.ipynb);
[Module 43](43_llms_ai_agents.ipynb)). As always in this course, a predicted potency is a hypothesis to be tested at the
bench, not a fact.

---
## 📌 Key takeaways

- **A molecule has no single feature vector.** Descriptors are readable but coarse; **circular fingerprints** mark atom
  environments and are the QSAR workhorse; **graphs** let a GNN learn the representation. **Tanimoto** similarity on
  fingerprints underlies similarity search and the similar-property principle.
- **Potency is logarithmic** (pIC50): an RMSE of 1.0 is a ten-fold error. Lipinski's rule of five is a soft drug-likeness
  filter, not a law.
- **On a random split** ridge, random forest and a compact MPNN all reached R² ≈ 0.5–0.6 here, below the noise ceiling
  (≈ 0.9); a tuned random forest on fingerprints is a hard baseline to beat.
- **The split decides what you measure.** The *same* model went from a healthy R² on a random split to **negative** R² on
  a **scaffold** split — the honest test of generalisation to new chemistry. Report the split that matches your deployment
  question; beware near-duplicate leakage.
- **Know your applicability domain.** Error grows as molecules get less similar to the training set, so similarity-to-train
  is a cheap reliability flag. Ensemble (tree) disagreement tracks error only weakly — "confident" ≠ "correct" — which
  motivates **conformal prediction** ([Module 42](42_trustworthy_ml.ipynb)).
- **Activity cliffs** (near-identical molecules, very different potency) break fingerprint models: our simulator makes a
  donor at site 0 vs another site produce the *same* fingerprint but a > 1-unit potency gap, and the model's error on cliff
  molecules is several-fold higher.
- **Interpretation can recover mechanism — carefully.** A transparent model on interpretable features rediscovered the
  documented rules (donor at site 0 helps, bulky at site 2 hurts), confirmed in sign and roughly in size by the oracle's
  controlled toggles; but correlation-based importances mix direct and indirect effects (and even a single toggle can move
  two rules at once).
- **Screening is ranking.** Enrichment factor and ROC/PR at the top of the list matter more than average error, and a real
  candidate must satisfy **multiple objectives** (potency *and* solubility, ADMET, synthesisability).

---
## ✅ Checkpoint

1. Read the SMILES `c1ccncc1C(=O)N`. How many ring atoms, is the ring aromatic, what is the nitrogen in the ring, and what is the `C(=O)N` group?
   <details><summary>Answer</summary>Six ring atoms written in lower case → an **aromatic** six-membered ring with one ring nitrogen: **pyridine**. The `C(=O)N` substituent is a carbon double-bonded to O and single-bonded to N — an **amide** (carboxamide).</details>
2. Two fingerprints have 30 and 26 bits set, sharing 22. What is their Tanimoto similarity?
   <details><summary>Answer</summary>$|a\cap b| = 22$; $|a\cup b| = 30 + 26 - 22 = 34$; Tanimoto $= 22/34 \approx 0.65$.</details>
3. Why is a scaffold split usually harder than a random split, and when would a random split be the fair one?
   <details><summary>Answer</summary>A scaffold split holds out whole chemical series, so test molecules are dissimilar to training ones (extrapolation). A random split leaves close analogues on both sides (interpolation). A random split is fair only when you will deploy on chemistry closely related to your training set.</details>
4. A model has EF1% = 8 on a library where 5 % of molecules are active. What fraction of the top 1 % are active, and why can EF1% exceed 1/base-rate?
   <details><summary>Answer</summary>Fraction active in the top 1 % $= 8 \times 0.05 = 0.40$ (40 %). EF is capped at $1/\text{base rate} = 20$ here (if the whole top 1 % were active); EF1% = 8 means the top 1 % is eight-fold enriched over random.</details>
5. What is an activity cliff, and why can't a fingerprint model predict both members of a cliff correctly?
   <details><summary>Answer</summary>A pair of very similar molecules with very different activity. If their fingerprints are (nearly) identical, the model maps them to (nearly) the same prediction, so it must be wrong about at least one — it lacks the information (e.g. *where* a group sits) that causes the difference.</details>
6. Your random forest predicts pIC50 = 8.5 for a molecule whose nearest training neighbour has Tanimoto 0.2. Should you trust it? What would you report alongside the prediction?
   <details><summary>Answer</summary>No — the molecule is far outside the applicability domain (nearest neighbour 0.2), where error is large and uncertainty poorly calibrated. Report the max similarity to the training set, a calibrated/conformal interval, and flag it as an extrapolation.</details>
7. Why is average R² a poor metric for a virtual-screening model, and what should you report instead?
   <details><summary>Answer</summary>Screening cares about the *top* of the ranked list, not average error; a model with mediocre R² can rank actives well. Report enrichment factor (EF1%/EF5%), ROC-AUC and especially PR-AUC (actives are rare), and ideally early-recognition metrics like BEDROC.</details>
8. Code reading: `cv_predictions(fp, rf_fn, groups=mols.scaffold.values)` uses `GroupKFold`. What does passing `groups` change compared with `KFold`, and why does it matter for molecules?
   <details><summary>Answer</summary>`GroupKFold` guarantees that all molecules sharing a scaffold fall in the *same* fold, so no scaffold is split across train and test. This prevents the leakage of near-identical analogues that would make a random `KFold` overstate performance.</details>

## 🧠 Questions & discussion

1. A paper reports R² = 0.9 for potency prediction on a random split. What three questions would you ask before believing the model will help a real project, and which experiment (Section 4/5) would you run first?
2. Activity cliffs are where prediction is both hardest and most valuable (telling a potent analogue from its inactive twin). Does it make sense to *weight* cliffs more heavily in training and evaluation, and what could go wrong if you do?
3. When is a graph neural network worth its extra cost over a random forest on fingerprints for molecular property prediction, and when is it not? (Think about data-set size, the split, and the kind of structure–activity relationship.)
4. A model confidently predicts high potency for a molecule far outside its applicability domain. Whose responsibility is the wasted synthesis — the modeller, the chemist, or the tool? How should uncertainty be communicated to prevent it?
5. Models that predict toxicity can, run in reverse, suggest *more* toxic molecules (as discussed in the design and agents modules). What governance would you want around sharing potent-by-design or toxicity models, and how does it differ for a predictor versus a generator?

## 🚀 Bonus — going further

- Add **conformal prediction** (a preview of [Module 42](42_trustworthy_ml.ipynb)): hold out a calibration set, take the quantile of absolute residuals, and turn every prediction into an interval with ~90 % coverage. Check the empirical coverage on the test set, and whether intervals widen out of distribution.
- Build a **Δ-learning (pairwise) model** for activity cliffs: train a model to predict the potency *difference* between two similar molecules from the difference of their features, and see whether it handles cliffs better than the absolute model.
- Map real **fingerprint-bit importances** back to atom environments: with RDKit (outside this course's dependencies) compute ECFP with bit provenance, take a random forest's top bits, and draw the substructures they correspond to — the production version of Section 7.
- Run the **scaffold-split MPNN** properly: loop a `GroupKFold` over scaffolds, fit `train_mpnn` on each fold, and compare the graph model's grouped R² with the random forest's. Does the graph model generalise across chemistry any better?
- Try a production stack on real data (outside this course): RDKit fingerprints + scikit-learn, or Chemprop, on a ChEMBL or MoleculeNet target, with a scaffold split — and compare the honest numbers with the random-split ones.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 40 · Molecular machine learning for drug discovery
Biological question: predict potency/solubility of unseen molecules well enough to prioritise synthesis — and know
                     when not to trust the prediction
Data (simulated): simulate_molecules (10 scaffolds × 15 substituents; documented SAR rules; noise-free oracle;
                  valid SMILES)
Method: fingerprint + ridge / random forest; compact sum-pooling MPNN (Module 28); grouped (scaffold) & time splits;
        applicability domain (max Tanimoto); ensemble uncertainty; activity-cliff analysis; design-matrix importances;
        virtual screening (EF, ROC/PR); multi-task network
Key equation: pIC50 = −log10(IC50);  Tanimoto = |a∩b|/|a∪b|;  EF_x% = (active fraction in top x%)/(base rate)
Key parameters: representation (descriptor/fingerprint/graph), split (random/scaffold/time), RF max_features,
                MPNN depth, active threshold (pIC50 > 7)
Ground-truth check result: random-split R² ridge/RF/MPNN ≈ 0.58/0.54/0.62 (ceiling ≈ 0.9); scaffold-split R²
                           ≈ 0.2 (MPNN), ≈ 0 or below (fingerprint models); error rises as similarity-to-train falls;
                           donor@0 effect ≈ +1.4 and bulky@2 ≈ −0.8 (group means) vs oracle +1.3/−1.3;
                           EF1% a few-fold (small denominator); RF error on cliffs ≈ 3× the library average
What changed when I modified … (split / representation / RF max_features / multi-task auxiliary): …
One failure mode: confident predictions on out-of-domain (scaffold-split) molecules; fingerprint models blind to
                  activity cliffs
Connection to research: Rogers & Hahn 2010 (ECFP); Sheridan 2013 / van Tilborg 2022 (splits, cliffs);
                        Stokes 2020 / Wong 2024 (antibiotics); Yang 2019 (Chemprop); Huang 2021 (TDC)
```

## 📚 References

- Weininger D. (1988) SMILES, a chemical language and information system. *J Chem Inf Comput Sci* 28:31–36.
- Hansch C., Fujita T. (1964) ρ-σ-π analysis. A method for the correlation of biological activity and chemical structure. *J Am Chem Soc* 86:1616–1626.
- Lipinski C. A., Lombardo F., Dominy B. W., Feeney P. J. (1997) Experimental and computational approaches to estimate solubility and permeability. *Adv Drug Deliv Rev* 23:3–25 (the rule of five).
- Bemis G. W., Murcko M. A. (1996) The properties of known drugs: 1. Molecular frameworks. *J Med Chem* 39:2887–2893.
- Rogers D., Hahn M. (2010) Extended-connectivity fingerprints. *J Chem Inf Model* 50:742–754.
- Wu Z. *et al.* (2018) MoleculeNet: a benchmark for molecular machine learning. *Chem Sci* 9:513–530.
- Yang K. *et al.* (2019) Analyzing learned molecular representations for property prediction. *J Chem Inf Model* 59:3370–3388 (Chemprop).
- Sheridan R. P. (2013) Time-split cross-validation as a method for estimating the goodness of prospective prediction. *J Chem Inf Model* 53:783–790.
- Tropsha A. (2010) Best practices for QSAR model development, validation, and exploitation. *Mol Inform* 29:476–488.
- van Tilborg D., Alenicheva A., Grisoni F. (2022) Exposing the limitations of molecular machine learning with activity cliffs. *J Chem Inf Model* 62:5938–5951.
- Truchon J.-F., Bayly C. I. (2007) Evaluating virtual screening methods: good and bad metrics for the 'early recognition' problem. *J Chem Inf Model* 47:488–508 (BEDROC).
- Mayr A., Klambauer G., Unterthiner T., Hochreiter S. (2016) DeepTox: toxicity prediction using deep learning. *Front Environ Sci* 3:80.
- Lakshminarayanan B., Pritzel A., Blundell C. (2017) Simple and scalable predictive uncertainty estimation using deep ensembles. *NeurIPS*.
- Stokes J. M. *et al.* (2020) A deep learning approach to antibiotic discovery. *Cell* 180:688–702 (halicin).
- Wong F. *et al.* (2024) Discovery of a structural class of antibiotics with explainable deep learning. *Nature* 626:177–185.
- Lyu J. *et al.* (2019) Ultra-large library docking for discovering new chemotypes. *Nature* 566:224–229.
- Huang K. *et al.* (2021) Therapeutics Data Commons: machine learning datasets and tasks for drug discovery and development. *NeurIPS Datasets & Benchmarks*.
- Wildman S. A., Crippen G. M. (1999) Prediction of physicochemical parameters by atomic contributions. *J Chem Inf Comput Sci* 39:868–873 (the logP model our descriptor mimics).

## ➡️ Next

We have measured potency and flagged where a model should not be trusted, but we have not yet given it **calibrated**
uncertainty with guarantees, nor confronted the distribution shift that a scaffold split only hinted at.
**[Module 41 · Perturbation biology and causal inference](41_perturbation_causal_inference.ipynb)** turns from
prediction to *intervention*, and **[Module 42 · Trustworthy biological ML](42_trustworthy_ml.ipynb)** builds leakage
detection, calibration and conformal prediction — the guarantees this notebook kept pointing to.

In [ ]:
# ── Total run time ──
print(f"⏱ Notebook compute: {time.time() - T_START:.0f} s on {torch.get_num_threads()} CPU thread(s).")